# H17: Initial Gradient Anisotropy as a Predictor of Muon Architecture Benefit

**Counterpart script:** `run_experiment.py` in the same directory.

## Pair-level identity
This notebook is the presentation and analysis counterpart to the script. It intentionally **imports and uses the script implementation** rather than silently re-implementing the training logic inside the notebook.

## Current question
Across the tested architectures, does **initial gradient anisotropy** predict how much Muon outperforms momentum SGD on final training loss?

## Important limitation
Despite the historical directory name, the current implementation does **not** directly measure a separate update-direction-quality metric. It therefore does **not** settle a literal **"direction vs conditioning"** mechanism split. The present experiment is narrower: it evaluates **initial gradient anisotropy as a predictor of Muon benefit** in this controlled setup.

## 1. Notebook-safe imports and script loading

The next cell resolves the repository root explicitly, loads the paired script as a module, and avoids notebook-local copies of the experiment logic.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise RuntimeError("matplotlib is required to render the analysis figures in this notebook.") from exc

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    pass

TARGET_REL = Path("experiments/Tier_4_Falsification_Stress_Tests/H17_DIRECTION_VS_CONDITIONING_PREDICTOR/run_experiment.py")


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / TARGET_REL).exists():
            return candidate
        if candidate.name == "H17_DIRECTION_VS_CONDITIONING_PREDICTOR" and (candidate / "run_experiment.py").exists():
            return candidate.parents[2]
    raise FileNotFoundError(
        "Could not resolve the repository root. Run this notebook from inside the repository or update TARGET_REL."
    )


REPO_ROOT = resolve_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / TARGET_REL.parent
SCRIPT_PATH = REPO_ROOT / TARGET_REL
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location("h17_run_experiment", SCRIPT_PATH)
h17 = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h17)

print(f"Repository root: {REPO_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Script path: {SCRIPT_PATH}")

## 2. Notebook utilities

These helpers are presentation-only: table display, record extraction, and lightweight formatting. The numerical experiment remains owned by the script module.

In [ ]:
def as_dataframe(rows, index=None):
    if pd is None:
        return rows
    df = pd.DataFrame(rows)
    if index is not None:
        df = df.set_index(index)
    return df



def display_rows(rows, title=None, index=None, precision=4):
    if title:
        print(title)
    if pd is not None:
        df = as_dataframe(rows, index=index)
        display(df.round(precision))
    else:
        for row in rows:
            formatted = {}
            for key, value in row.items():
                if isinstance(value, float) and np.isfinite(value):
                    formatted[key] = round(value, precision)
                else:
                    formatted[key] = value
            print(formatted)



def collect_final_loss_rows(results):
    rows = []
    for arch in results["config"]["architectures"]:
        arch_result = results["architectures"][arch]
        for optimizer in ["muon", "sgd"]:
            summary = arch_result["final_training"][optimizer]
            for seed_record in summary["seed_records"]:
                rows.append({
                    "architecture": arch,
                    "optimizer": optimizer,
                    "data_seed": seed_record["data_seed"],
                    "final_loss": seed_record["final_loss"],
                    "is_finite": seed_record["is_finite"],
                    "selected_lr": summary["lr"],
                })
    return rows



def collect_advantage_rows(results):
    rows = []
    for arch in results["config"]["architectures"]:
        for record in results["architectures"][arch]["per_seed_advantage_records"]:
            rows.append({
                "architecture": arch,
                "data_seed": record["data_seed"],
                "muon_loss": record["muon_loss"],
                "sgd_loss": record["sgd_loss"],
                "both_finite": record["both_finite"],
                "advantage": record["advantage"] if record["advantage"] is not None else np.nan,
            })
    return rows

## 3. Reproducibility, run mode, and runtime budget

The script supports both a full run and a smoke configuration. By default this notebook uses the **full** configuration so that it matches the normal script entrypoint. For quick verification, set `RUN_MODE = "smoke"` before executing the next cell.

In [ ]:
import os

RUN_MODE = globals().get("RUN_MODE") or os.environ.get("H17_NOTEBOOK_RUN_MODE", "full")
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("RUN_MODE must be either 'full' or 'smoke'.")

RUN_KWARGS = {}
if RUN_MODE == "smoke":
    RUN_KWARGS = {
        "num_steps": 50,
        "num_seeds": 2,
        "lr_sweep_seed_count": 2,
    }

planned_config = h17.build_config(**RUN_KWARGS)
known_runtime_note = (
    "Known from prior script verification: the default full configuration took about 88 seconds on this machine; "
    "the reduced smoke configuration is much faster."
)

print("Run mode:", RUN_MODE)
print("Smoke override env var:", "H17_NOTEBOOK_RUN_MODE")
print("Title: H17: Initial Gradient Anisotropy as a Predictor of Muon Architecture Benefit")
print("Architectures:", planned_config["architectures"])
print("Seeds:", planned_config["seeds"])
print("LR sweep seeds:", planned_config["lr_sweep_seeds"])
print("Muon LR grid:", planned_config["lr_muon"])
print("SGD LR grid:", planned_config["lr_sgd"])
print(
    f"Network: {planned_config['num_layers']}-layer, "
    f"{planned_config['dim']}x{planned_config['dim']}, batch={planned_config['batch_size']}"
)
print("Momentum:", planned_config["momentum"])
print("Newton-Schulz iterations:", planned_config["ns_iters"])
print("Estimated train() calls:", planned_config["estimated_train_calls"])
print(known_runtime_note)


## 4. Execute the script-backed experiment

This is the only place where the numerical experiment is run. All later sections read from the returned structured results.

In [ ]:
results = h17.run_experiment(verbose=False, **RUN_KWARGS)
h17.print_summary(results)

summary_rows = results["summary_rows"]
display_rows(
    summary_rows,
    title="Architecture-level summary:",
    index="architecture",
    precision=4,
)

### Interpretation

At this point the notebook has reproduced the script-backed experiment and exposed the structured outputs needed for inspection. The remaining sections are presentation and interpretation layers only.

## 5. Architecture anisotropy diagnostics

The first figure shows **per-layer** anisotropy for one representative seed per architecture. The second figure shows the **per-seed mean anisotropy** across layers for each architecture. These are spectral diagnostics of the *initial gradients*; they are **not** direct measurements of Muon's update-direction quality.

In [ ]:
archs = results["config"]["architectures"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, arch in zip(axes.flat, archs):
    diagnostic = results["architectures"][arch]["representative_diagnostic"]
    layer_stats = diagnostic["layer_stats"]
    layers = [layer["layer"] + 1 for layer in layer_stats]
    anisos = [layer["anisotropy"] for layer in layer_stats]
    ax.bar(layers, anisos, color="C0", alpha=0.8)
    ax.set_title(f"{arch} | representative data seed {diagnostic['data_seed']}")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Gradient anisotropy")
plt.suptitle("Representative per-layer initial gradient anisotropy", y=1.02)
plt.tight_layout()
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
for idx, arch in enumerate(archs):
    values = results["architectures"][arch]["anisotropy_per_seed"]
    mean_value = results["architectures"][arch]["anisotropy_mean"]
    std_value = results["architectures"][arch]["anisotropy_std"]
    jitter = np.linspace(-0.06, 0.06, len(values)) if len(values) > 1 else np.array([0.0])
    ax.scatter(np.full(len(values), idx) + jitter, values, s=50, color="C0", alpha=0.75)
    ax.errorbar(idx, mean_value, yerr=std_value, fmt="_", color="black", capsize=4, markersize=24)
ax.set_xticks(range(len(archs)))
ax.set_xticklabels(archs, rotation=20)
ax.set_ylabel("Mean anisotropy across layers")
ax.set_title("Per-seed anisotropy with architecture means ± sample std")
plt.tight_layout()
plt.show()
plt.close(fig)

anisotropy_rows = [
    {
        "architecture": row["architecture"],
        "anisotropy_mean": row["anisotropy_mean"],
        "anisotropy_std": row["anisotropy_std"],
    }
    for row in results["summary_rows"]
]
display_rows(anisotropy_rows, title="Anisotropy summary:", index="architecture", precision=3)

### Interpretation

These diagnostics justify the predictor variable actually used by the experiment: **initial gradient anisotropy**. They should be read as descriptive summaries of the gradient spectrum at initialization, not as evidence for or against a separate direction-quality mechanism.

## 6. Learning-rate sweep comparison by architecture and optimizer

For each architecture and optimizer, the script sweeps a fixed LR grid on the first `lr_sweep_seed_count` seeds, computes the mean over **finite** final losses, and selects the best LR. Divergent runs are reported explicitly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, arch in zip(axes.flat, archs):
    arch_result = results["architectures"][arch]
    for optimizer, color, marker in [("muon", "C0", "o"), ("sgd", "C1", "s")]:
        sweep = arch_result["lr_sweeps"][optimizer]
        records = sorted(sweep["records"], key=lambda record: record["lr"])
        lrs = [record["lr"] for record in records]
        means = [record["mean_finite_loss"] for record in records]
        divs = [record["divergent_count"] for record in records]
        ax.plot(lrs, means, marker=marker, color=color, label=optimizer)
        best_lr = sweep["best_lr"]
        best_record = min(records, key=lambda record: abs(record["lr"] - best_lr))
        ax.scatter([best_record["lr"]], [best_record["mean_finite_loss"]], s=120, color=color, edgecolor="black", linewidth=1.0)
        for lr, mean_loss, divergent_count in zip(lrs, means, divs):
            if divergent_count:
                ax.annotate(
                    f"{divergent_count} div",
                    (lr, mean_loss),
                    textcoords="offset points",
                    xytext=(0, 7),
                    ha="center",
                    fontsize=8,
                )
    ax.set_xscale("log")
    ax.set_title(arch)
    ax.set_xlabel("Learning rate")
    ax.set_ylabel("Mean finite final loss")
    ax.legend()
plt.suptitle("Learning-rate sweep outcomes", y=1.02)
plt.tight_layout()
plt.show()
plt.close(fig)

lr_selection_rows = []
for row in results["summary_rows"]:
    lr_selection_rows.append({
        "architecture": row["architecture"],
        "muon_best_lr": row["muon_best_lr"],
        "sgd_best_lr": row["sgd_best_lr"],
    })
display_rows(lr_selection_rows, title="Selected best learning rates:", index="architecture", precision=4)

### Interpretation

The LR sweep is part of the experimental protocol, not the main scientific claim. It establishes a fairer architecture-by-architecture optimizer comparison than using a single global LR, but it is still a small fixed-grid search rather than a full hyperparameter study.

## 7. Final loss comparison and Muon advantage

The figure below reports the final-loss outcomes using the selected LR for each optimizer and architecture. Finite losses are shown at the seed level, and divergence counts are retained explicitly rather than hidden.

In [ ]:
final_loss_rows = collect_final_loss_rows(results)
advantage_rows = collect_advantage_rows(results)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
loss_ax, adv_ax = axes

for idx, arch in enumerate(archs):
    arch_result = results["architectures"][arch]
    for offset, optimizer, color in [(-0.16, "muon", "C0"), (0.16, "sgd", "C1")]:
        summary = arch_result["final_training"][optimizer]
        finite_losses = [record["final_loss"] for record in summary["seed_records"] if record["is_finite"]]
        if finite_losses:
            jitter = np.linspace(-0.05, 0.05, len(finite_losses)) if len(finite_losses) > 1 else np.array([0.0])
            xs = np.full(len(finite_losses), idx + offset) + jitter
            loss_ax.scatter(xs, finite_losses, s=55, color=color, alpha=0.8)
            loss_ax.hlines(summary["mean_finite_loss"], idx + offset - 0.09, idx + offset + 0.09, color="black", linewidth=2)
        y_text = summary["mean_finite_loss"] if np.isfinite(summary["mean_finite_loss"]) else 0.0
        loss_ax.text(idx + offset, y_text, f"div={summary['divergent_count']}", fontsize=8, ha="center", va="bottom")

    arch_advantages = [
        record["advantage"]
        for record in arch_result["per_seed_advantage_records"]
        if record["advantage"] is not None and np.isfinite(record["advantage"])
    ]
    if arch_advantages:
        jitter = np.linspace(-0.05, 0.05, len(arch_advantages)) if len(arch_advantages) > 1 else np.array([0.0])
        adv_ax.scatter(np.full(len(arch_advantages), idx) + jitter, arch_advantages, s=55, color="C2", alpha=0.8)
        adv_ax.hlines(
            arch_result["mean_finite_per_seed_advantage"],
            idx - 0.1,
            idx + 0.1,
            color="black",
            linewidth=2,
        )

loss_ax.set_xticks(range(len(archs)))
loss_ax.set_xticklabels(archs, rotation=20)
loss_ax.set_ylabel("Final loss")
loss_ax.set_title("Final losses by architecture and optimizer")

adv_ax.set_xticks(range(len(archs)))
adv_ax.set_xticklabels(archs, rotation=20)
adv_ax.set_ylabel("Per-seed Muon advantage (SGD loss / Muon loss)")
adv_ax.set_yscale("log")
adv_ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
adv_ax.set_title("Per-seed Muon advantage")

plt.tight_layout()
plt.show()
plt.close(fig)

final_summary_rows = []
for row in results["summary_rows"]:
    final_summary_rows.append({
        "architecture": row["architecture"],
        "muon_mean_finite_loss": row["muon_mean_finite_loss"],
        "sgd_mean_finite_loss": row["sgd_mean_finite_loss"],
        "muon_divergent_count": row["muon_divergent_count"],
        "sgd_divergent_count": row["sgd_divergent_count"],
        "advantage_ratio_of_means": row["advantage_ratio_of_means"],
        "mean_finite_per_seed_advantage": row["mean_finite_per_seed_advantage"],
    })
display_rows(final_summary_rows, title="Final-loss and advantage summary:", index="architecture", precision=4)

### Interpretation

The main performance quantity is the **architecture-level Muon advantage**, defined in the script as the ratio of the SGD mean finite loss to the Muon mean finite loss. The seed-level scatter is included because the architecture-level ratio can hide variability and because divergence is an important part of optimizer behavior in this setup.

## 8. Predictor test: anisotropy vs Muon advantage

This is the central architecture-level relationship tested by the current implementation. Because there are only four architecture-level points in the default run, the fitted line and Pearson correlation should be interpreted as **descriptive evidence within this setup**, not as a decisive predictive law.

In [ ]:
corr = results["correlation_analysis"]
x = np.array(corr["anisotropy_values"], dtype=float)
y = np.array(corr["advantage_values"], dtype=float)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, s=90, color="C3")
for arch, xi, yi in zip(archs, x, y):
    ax.annotate(arch, (xi, yi), textcoords="offset points", xytext=(6, 6))

if len(x) > 1 and np.all(np.isfinite(x)) and np.all(np.isfinite(y)):
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, slope * x_line + intercept, linestyle="--", color="black", label="Linear fit")

ax.set_xlabel("Mean initial gradient anisotropy")
ax.set_ylabel("Muon advantage (SGD mean loss / Muon mean loss)")
ax.set_title(f"Architecture-level anisotropy vs Muon advantage (r = {corr['pearson_r']:.3f})")
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)

### Explicit caveat

A positive anisotropy/advantage relationship would support the narrower claim that **anisotropy is a useful predictor in this experiment**. It would **not** demonstrate that conditioning is the dominant mechanism, nor would a weak relationship prove that update direction dominates. A genuine direction-vs-conditioning comparison still requires a second, separately measured direction-quality metric.

## 9. Final verdict and limitations

The next cell converts the returned script outputs into a concise end-of-notebook research summary.

In [ ]:
corr = results["correlation_analysis"]
if corr["t1_positive_correlation_gt_0_5"]:
    predictor_statement = (
        "In this run, higher initial gradient anisotropy is positively associated with larger Muon benefit "
        "across the tested architectures."
    )
else:
    predictor_statement = (
        "In this run, initial gradient anisotropy is not strongly positively associated with Muon benefit "
        "across the tested architectures."
    )

verdict_md = f"""
### Final verdict

- **Pearson correlation:** `{corr['pearson_r']:.3f}`
- **T1 (r > 0.5):** `{'PASS' if corr['t1_positive_correlation_gt_0_5'] else 'FAIL'}`
- **T2 (same argmax architecture):** `{'PASS' if corr['t2_same_argmax_architecture'] else 'FAIL'}`
- **Highest-anisotropy architecture:** `{corr['max_anisotropy_architecture']}`
- **Highest-advantage architecture:** `{corr['max_advantage_architecture']}`

**Interpretation:** {predictor_statement}

**Scope of the claim:** this notebook supports only the claim that *initial gradient anisotropy may predict Muon benefit in this setup*. It does **not** directly measure update-direction quality and therefore does **not** resolve a literal direction-vs-conditioning mechanism split.
"""

display(Markdown(verdict_md))

limitations_md = "\n".join([f"- {item}" for item in results["limitations"]])
display(Markdown("### Current limitations\n" + limitations_md))

## Next serious follow-up

To turn this historical directory name into a literal experiment about **direction vs conditioning**, the next implementation step would be to add a **separate direction-quality metric** and compare its predictive power against the current anisotropy metric. Until then, the academically accurate description of this pair is:

> **Initial gradient anisotropy as a predictor of Muon architecture benefit across a small architecture panel.**